# Finetuning on GERA + LORuGEC


In [1]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

project root: /home/temari/god please no diploma/restore_punct


## Run config

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

LORUGEC_UPSAMPLE = 13
lorugec_files = [f"{DATA_DIR}/train_lorugec.json"] * LORUGEC_UPSAMPLE


cfg = RunConfig(
    name          = "04a_gera_lorugec_mixed",
    architecture  = "bert+crf",
    loss          = "crf",
    train_files   = [
        f"{DATA_DIR}/train_all_rare_punct.json",
        f"{DATA_DIR}/train_gera.json",
        *lorugec_files,
    ],
    val_files     = [f"{DATA_DIR}/val_gera.json"],
    num_epochs    = 3,
    learning_rate = 2e-5,
    crf_init_transitions = True,
    tags          = {"stage": "4-gera-lorugec", "strategy": "mix"},
)



print(cfg)

RunConfig(name='04a_gera_lorugec_mixed', architecture='bert+crf', loss='crf', train_files=['/home/temari/god please no diploma/restore_punct/data/train_all_rare_punct.json', '/home/temari/god please no diploma/restore_punct/data/train_gera.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.

## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)

[04a_gera_lorugec_mixed] architecture=bert+crf loss=crf epochs=3 lr=2e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[04a_gera_lorugec_mixed] computing empirical CRF transitions from ['/home/temari/god please no diploma/restore_punct/data/train_all_rare_punct.json', '/home/temari/god please no diploma/restore_punct/data/train_gera.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/117204 [00:00<?, ? examples/s]

Map:   0%|          | 0/747 [00:00<?, ? examples/s]

/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,38.334346,2.171292,0.960743,0.961228,0.959968,0.961228
2,34.511069,2.108216,0.961650,0.962797,0.961652,0.962797
3,31.103125,2.105956,0.962298,0.962954,0.962037,0.962954


Model saved -> /home/temari/god please no diploma/restore_punct/models/04a_gera_lorugec_mixed


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    ma = rep.get('macro avg', {})
    print(f"{test_name:>14s} | W-F1={wa.get('f1-score', 0):.4f}  M-F1={ma.get('f1-score', 0):.4f}  P={wa.get('precision', 0):.4f}  R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[04a_gera_lorugec_mixed] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[04a_gera_lorugec_mixed] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[04a_gera_lorugec_mixed] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 04a_gera_lorugec_mixed)
Wrote /home/temari/god please no diploma/restore_punct/results/04a_gera_lorugec_mixed.json
Wrote /home/temari/god please no diploma/restore_punct/results/04a_gera_lorugec_mixed.xlsx
  General_Test | W-F1=0.9527  M-F1=0.4543  P=0.9517  R=0.9549
     GERA_Test | W-F1=0.9640  M-F1=0.5035  P=0.9645  R=0.9656
  LORuGEC_Test | W-F1=0.9729  M-F1=0.5237  P=0.9728  R=0.9735


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела.

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань уже".

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



## Save stats

In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 11
  Yandex runs : 0


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'